# Step 5: LangGraph — From LangChain Chains to Stateful Agents

This notebook is Step 5 of the LangChain learning roadmap, following the Colab / Groq workflow from Steps 1-4.

Adapted from [DataCamp's LangGraph Agents tutorial](https://www.datacamp.com/tutorial/langgraph-agents), rewritten to use:
- **ChatGroq** (`llama-3.3-70b-versatile`) instead of OpenAI/Gemini
- **Google Colab** with Colab Secrets for the API key
- The **weather (wttr.in)** and **calculator** tools built in Step 3

Only the modern, currently-recommended LangGraph patterns are covered. LangSmith tracing, deprecated agent patterns, and multi-agent/subgraph setups are intentionally skipped, since they are outside the scope of this step.

## 1. What is LangGraph, and how is it different from LangChain?

**LangChain** is great for building *linear* pipelines: prompt to LLM to parser to output (a "chain"). Once you need loops, branching decisions, or an agent that calls a tool, checks the result, and decides what to do next, a straight chain isn't the right shape.

**LangGraph** solves this by modeling the agent as a **graph**: `nodes` are steps (an LLM call, a tool call, a function), and `edges` (including *conditional* edges) decide which node runs next based on the current `state`. This is what lets an agent loop ("call a tool, look at the result, decide to call another tool or stop") instead of running once and finishing.

In short: LangChain = build the pieces (prompts, tools, models). LangGraph = wire those pieces into a stateful, possibly cyclical workflow.

### Setup

Installing the packages needed for this notebook, and setting up the Groq API key from **Colab Secrets** (Secrets panel in the left sidebar of Colab, key icon), instead of a `.env` file or hardcoded key.

In [ ]:
# Install the required packages (quiet mode to keep the output clean)
!pip install -q langchain langchain-groq langgraph requests

In [ ]:
# Imports for this notebook
from typing import TypedDict, Annotated, List
import operator
import requests

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver

# Pull the Groq API key from Colab Secrets instead of a .env file
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Our LLM for this notebook - Groq's llama-3.3-70b-versatile
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

## 2. StateGraph and the state (`TypedDict`)

The **state** is the shared piece of data that flows through every node in the graph. Each node reads the current state, does something, and returns an update to it.

We define the shape of the state with a `TypedDict` (a dictionary with declared, type-checked keys). For a chat-style agent, the state mainly needs to hold the running list of messages.

Notice the `Annotated[list[BaseMessage], operator.add]` part: this is a **reducer**. Instead of a node's returned messages *replacing* the existing list, the reducer tells LangGraph to *append* new messages onto the old ones, so conversation history is preserved across every step of the graph.

`StateGraph` is the class that ties this `AgentState` shape to a graph of nodes and edges - it is the "fusion" of a plain `Graph` and the `State`.

In [ ]:
# Defining the shape of our shared state using TypedDict
class AgentState(TypedDict):
    # messages is a list of chat messages; operator.add means new messages
    # get appended to this list instead of overwriting it
    messages: Annotated[List[BaseMessage], operator.add]

## 3. Nodes — plain Python functions

A **node** is just a Python function that:
1. takes the current `state` as input,
2. does some work (call an LLM, run some logic),
3. returns a dictionary with the keys of the state that changed.

Below is a simple node that sends the current messages to our Groq LLM and returns its reply. This is the same idea as the `process` node in the tutorial, just pointed at `ChatGroq` instead of `ChatOpenAI`.

In [ ]:
# A node is a plain Python function: state in, partial state update out
def call_model(state: AgentState) -> AgentState:
    # Send the full running conversation to the LLM
    response = llm.invoke(state["messages"])
    # Return only the new message; the reducer will append it to the list
    return {"messages": [response]}

## 4. Edges — fixed edges, conditional edges, `START` and `END`

**Edges** decide what happens after a node runs:

- **Fixed edges** (`graph.add_edge(a, b)`) always go from node `a` straight to node `b`. No decision involved.
- **`START`** and **`END`** are special built-in markers for "where the graph begins" and "where the graph stops".
- **Conditional edges** (`graph.add_conditional_edges(...)`) run a small router function after a node, and that function's return value picks which node runs next. This is what makes loops possible: the router can send control back to an earlier node instead of always moving forward.

First, a minimal graph using only fixed edges, to see the mechanics before adding branching:

In [ ]:
# Build a minimal graph: START -> call_model -> END (fixed edges only)
graph_builder = StateGraph(AgentState)
graph_builder.add_node("call_model", call_model)   # register the node
graph_builder.add_edge(START, "call_model")         # fixed edge: entry point
graph_builder.add_edge("call_model", END)           # fixed edge: exit point

simple_app = graph_builder.compile()

# Visualize the compiled graph structure as ASCII
simple_app.get_graph().print_ascii()

In [ ]:
# Quick smoke test of the simple graph
result = simple_app.invoke({"messages": [HumanMessage(content="Say hello in one short sentence.")]})
print(result["messages"][-1].content)

## 5. Tools and `ToolNode` — the weather and calculator tools

A **tool** is a function the LLM can decide to call. Two things make a plain function into a LangGraph/LangChain tool:

- the **`@tool` decorator**, which registers it,
- a clear **docstring**, which is what the LLM actually reads to decide when and how to call the tool.

Below are the same two tools from Step 3 - weather (via `wttr.in`) and calculator - rewritten as proper `@tool`-decorated functions.

A **`ToolNode`** is a ready-made LangGraph node that takes a list of tools and knows how to execute whichever one the LLM asked for, without us writing that dispatch logic by hand.

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city.

    Args:
        city (str): Name of the city, e.g. 'Faisalabad' or 'Lahore'

    Returns:
        str: A short plain-text weather summary for that city.
    """
    # wttr.in's format=3 gives a compact one-line summary, e.g. "Faisalabad: 🌤 +32°C"
    response = requests.get(f"https://wttr.in/{city}?format=3", timeout=10)
    if response.status_code == 200:
        return response.text.strip()
    return f"Could not fetch weather for {city} (status {response.status_code})"


@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression.

    Args:
        expression (str): A math expression using +, -, *, /, and parentheses,
            e.g. '12 * (3 + 4)'

    Returns:
        str: The numeric result as a string, or an error message.
    """
    # Only allow digits, operators, parentheses, decimal points, and spaces -
    # keeps eval() restricted to arithmetic only, nothing else
    allowed_chars = set("0123456789+-*/(). ")
    if not set(expression) <= allowed_chars:
        return "Error: expression contains unsupported characters"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


# Collect our tools and let the LLM know they exist
tools = [get_weather, calculator]
llm_with_tools = llm.bind_tools(tools)

# ToolNode wraps our tool list into a ready-to-use graph node
tool_node = ToolNode(tools)

## 6. ReAct agent pattern — `StateGraph` with a tool-calling loop

This is the core pattern: an **agent node** calls the LLM (which may decide to call a tool), and a **conditional edge** checks the LLM's reply:

- if it contains tool calls, route to the `tools` node (our `ToolNode`), run the tool(s), then loop back to the agent node so the LLM can see the tool result and respond,
- if it doesn't contain tool calls, the LLM has given a final answer, so route to `END`.

That loop (agent to tools to agent to ... to END) is the ReAct ("Reason + Act") pattern, built here explicitly with `StateGraph` rather than the `create_react_agent` shortcut, so the wiring is visible.

In [ ]:
# Agent node: calls the tool-bound LLM with the running conversation
def agent_node(state: AgentState) -> AgentState:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# Router used by the conditional edge: decides where to go after agent_node
def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    # If the LLM's last message requested tool calls, go run the tools
    if getattr(last_message, "tool_calls", None):
        return "tools"
    # Otherwise the LLM gave a final answer, so we can stop
    return END


# Build the ReAct-style graph
react_builder = StateGraph(AgentState)
react_builder.add_node("agent", agent_node)
react_builder.add_node("tools", tool_node)

react_builder.add_edge(START, "agent")                 # fixed edge: entry point
react_builder.add_conditional_edges(                    # conditional edge: branches
    "agent",
    should_continue,
    {"tools": "tools", END: END},
)
react_builder.add_edge("tools", "agent")                # fixed edge: loop back to agent

react_app = react_builder.compile()

# Visualize the compiled ReAct graph structure as ASCII
react_app.get_graph().print_ascii()

In [ ]:
# Try the ReAct agent with both tools
result = react_app.invoke({
    "messages": [HumanMessage(content="What's the weather in Faisalabad, and what is 45 * 12?")]
})

for m in result["messages"]:
    print(f"{m.type}: {m.content}")

## 7. Persistent state and memory — `InMemorySaver`

By default, each `.invoke()` call is independent, so the agent forgets everything between calls. A **checkpointer** saves the state after every step so the graph can resume it later.

`InMemorySaver` is the simplest checkpointer: it keeps state in memory (lost when the Colab runtime restarts, fine for a learning notebook, not fine for production - that's what `SqliteSaver` or a database-backed checkpointer are for).

To use it, we compile the graph with `checkpointer=...`, and pass a `thread_id` in the config on every call. Every call sharing the same `thread_id` continues the same conversation.

In [ ]:
# Create an in-memory checkpointer and compile the SAME react graph with it
checkpointer = InMemorySaver()
memory_app = react_builder.compile(checkpointer=checkpointer)

# Visualize this compiled graph too - same structure, now with memory attached
memory_app.get_graph().print_ascii()

In [ ]:
# First turn of a conversation on thread "session_1"
config = {"configurable": {"thread_id": "session_1"}}

first = memory_app.invoke(
    {"messages": [HumanMessage(content="My name is Zariya. What's 10 + 5?")]},
    config,
)
print(first["messages"][-1].content)

# Second turn on the SAME thread_id - the agent should still remember the name
second = memory_app.invoke(
    {"messages": [HumanMessage(content="What is my name, and what did I just ask you to calculate?")]},
    config,
)
print(second["messages"][-1].content)

## Mapping LangGraph concepts to the from-scratch tool-calling loop (Step 3)

| From-scratch (Step 3) | LangGraph equivalent | What it replaces |
|---|---|---|
| The manual `while True:` loop that called the LLM, checked for a tool call, ran the tool, and appended the result before looping again | `StateGraph` + conditional edge (`should_continue`) looping `agent -> tools -> agent` | The hand-written loop and its exit condition are now expressed declaratively as graph edges instead of imperative control flow |
| The `if/elif` block that matched the tool name and called the right Python function | `ToolNode(tools)` | The manual tool-dispatch logic (matching names, unpacking args, calling the right function) is handled automatically once the tools are registered |
| A plain Python list (e.g. `messages = []`) that was manually appended to after every LLM call and tool call | `AgentState` (`TypedDict` with an `operator.add` reducer) | Manual list-append bookkeeping is replaced by the state + reducer, which merges new messages in automatically |
| Passing that list back into the next loop iteration, and starting over from scratch on every script run | `InMemorySaver` checkpointer + `thread_id` | Manual re-feeding of history within a single run is replaced by persistence across separate `.invoke()` calls (and potentially separate sessions) |
| A single, fixed sequence of steps hardcoded in the script | Fixed edges (`add_edge`) for the parts that always happen the same way (e.g. `START -> agent`, `tools -> agent`) | Hardcoded sequencing for the non-branching parts of the flow |
| The condition that decided "call a tool again" vs. "print the final answer and exit" | Conditional edges (`add_conditional_edges` with `should_continue`) | The branching decision itself, now a first-class part of the graph instead of buried in the loop's `if` statement |

The overall shape is the same tool-calling loop built in Step 3. LangGraph mainly gives that loop a name for each moving part (state, node, edge, ToolNode) and handles the bookkeeping (message accumulation, dispatch, persistence) that was previously written by hand.

## Practice exercise

Build your own simple LangGraph ReAct agent using **at least two tools of your choice** (not the weather/calculator tools above - pick something different, e.g. a unit converter, a word-counter, a currency lookup, a simple to-do tracker, etc.).

Steps to follow (same pattern as Section 6):
1. Define your tools with `@tool` and a clear docstring.
2. Bind them to `llm` with `.bind_tools(...)`.
3. Write an `agent_node` and a `should_continue` router.
4. Wire them into a `StateGraph` with `START`, `END`, and a conditional edge.
5. Compile the graph and call `.print_ascii()` to check the structure.
6. Test it with a prompt that should trigger both tools.

In [ ]:
# TODO: define your own tools here with @tool + docstrings
# @tool
# def my_tool_one(...) -> str:
#     """..."""
#     ...

# @tool
# def my_tool_two(...) -> str:
#     """..."""
#     ...

# TODO: bind your tools to the llm
# my_tools = [my_tool_one, my_tool_two]
# my_llm_with_tools = llm.bind_tools(my_tools)
# my_tool_node = ToolNode(my_tools)

# TODO: write your own agent_node and should_continue functions
# def my_agent_node(state: AgentState) -> AgentState:
#     ...

# def my_should_continue(state: AgentState) -> str:
#     ...

# TODO: build, compile, and visualize your graph
# my_builder = StateGraph(AgentState)
# ...
# my_app = my_builder.compile()
# my_app.get_graph().print_ascii()

# TODO: test it
# my_app.invoke({"messages": [HumanMessage(content="...")]})